# 2. SQL Analysis

## 2.1. Data Loading

2.1.1. Install and import all required R libraries

In [1]:
install.packages(c("sqldf", "googledrive"), quiet = TRUE)

library(sqldf)
library(googledrive)

also installing the dependencies ‘gsubfn’, ‘proto’, ‘RSQLite’, ‘chron’


Loading required package: gsubfn

Loading required package: proto

Warning message:
“no DISPLAY variable so Tk is not available”
Loading required package: RSQLite



2.1.2. Mount Google Drive to read the cleaned dataset

In [2]:
drive_auth(use_oob=TRUE)

Is it OK to cache OAuth access credentials in the folder ~/.cache/gargle
between R sessions?
1: Yes
2: No
Enter a number between 1 and 2, or enter 0 to exit.
Please point your browser to the following url: 

https://accounts.google.com/o/oauth2/v2/auth?client_id=603366585132-frjlouoa3s2ono25d2l9ukvhlsrlnr7k.apps.googleusercontent.com&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive%20https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email&redirect_uri=https%3A%2F%2Fwww.tidyverse.org%2Fgoogle-callback%2F&response_type=code&state=c64c23126141b3f11d250fc63ab3a9f2&access_type=offline&prompt=consent



In [4]:
base <- "North Star Case Study Dataset/cleaned/"

dl <- function(name) {
  drive_download(paste0(base, name), overwrite = TRUE)$local_path
}

# NOTE: stringsAsFactors tells R to keep text columns as strings, rather than converting them into categories
hubs <- read.csv(dl("hubs_clean.csv"), stringsAsFactors=FALSE)
customers <- read.csv(dl("customers_clean.csv"), stringsAsFactors=FALSE)
drivers <- read.csv(dl("drivers_clean.csv"), stringsAsFactors=FALSE)
vehicles <- read.csv(dl("vehicles_clean.csv"), stringsAsFactors=FALSE)
orders <- read.csv(dl("orders_clean.csv"), stringsAsFactors=FALSE)
deliveries <- read.csv(dl("deliveries_clean.csv"), stringsAsFactors=FALSE)
incidents <- read.csv(dl("incidents_clean.csv"), stringsAsFactors=FALSE)
complaints <- read.csv(dl("complaints_clean.csv"), stringsAsFactors=FALSE)
app_events <- read.csv(dl("app_events_clean.csv"), stringsAsFactors=FALSE)

File downloaded:

• hubs_clean.csv <id: 1DYLnMQnsQDEGyg1nl93Nxk0FoFBX-5Rk>

Saved locally as:

• hubs_clean.csv

File downloaded:

• customers_clean.csv <id: 1wvGKg18PTzOXvS3zgF7edFBumiPrKeo3>

Saved locally as:

• customers_clean.csv

File downloaded:

• drivers_clean.csv <id: 1ZGAHu6PfaoA1KVod2Cb318kH86o617z2>

Saved locally as:

• drivers_clean.csv

File downloaded:

• vehicles_clean.csv <id: 1JMEYq8khDKpkNLK9BoYFy42DSa8sL1qy>

Saved locally as:

• vehicles_clean.csv

File downloaded:

• orders_clean.csv <id: 13WWlNXhsfPXwHMTqwLxA16lrdcufljuB>

Saved locally as:

• orders_clean.csv

File downloaded:

• deliveries_clean.csv <id: 1i4v-y1YnzmtW9NJ-keXHiStL8eMKdqIV>

Saved locally as:

• deliveries_clean.csv

File downloaded:

• incidents_clean.csv <id: 1sqHskG5ZMhfTodFiSwuj53zU-Az892Ds>

Saved locally as:

• incidents_clean.csv

File downloaded:

• complaints_clean.csv <id: 11RcmLsZhFaydACSzhLnzPA2ZUHmL8ogD>

Saved locally as:

• complaints_clean.csv

File downloaded:

• app_events_cle

## 2.2. Delivery Status Breakdown

Count the number of each delivery status

In [ ]:
sqldf("
  SELECT delivery_status,
         COUNT(*) AS total
  FROM deliveries
  GROUP BY delivery_status
  ORDER BY total DESC
")

## 2.3. Complaint Volume by Type

Which complaint categories are raised most often

In [ ]:
sqldf("
  SELECT complaint_type,
         COUNT(*) AS total
  FROM complaints
  GROUP BY complaint_type
  ORDER BY total DESC
")

## 2.4. Hub Performance Summary

Per hub, determine delivery volume, failure count, failure rate and average customer rating

In [ ]:
sqldf("
  SELECT h.hub_name,
         COUNT(*) AS total_deliveries,
         SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed,
         ROUND(SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) * 100 / COUNT(*), 1) AS failed_pct,
         ROUND(AVG(d.customer_rating_post_delivery), 2) AS avg_rating
  FROM deliveries d
  JOIN hubs h ON d.hub_id = h.hub_id
  GROUP BY h.hub_name
  ORDER BY failed_pct DESC
")

## 2.5. Failed Delivery Rate by Zone

Which pickup zones have the worst faliure rate

In [ ]:
sqldf("
  SELECT o.pickup_zone,
         COUNT(*) AS total,
         SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed,
         ROUND(SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) * 100 / COUNT(*), 1) AS failed_pct
  FROM deliveries d
  JOIN orders o ON d.order_id = o.order_id
  GROUP BY o.pickup_zone
  ORDER BY failed_pct DESC
")

## 2.6. Service Type Complaint Rate

Determine complaint rate for each service type, to find out which service type generates the most dissatisfaction

In [ ]:
sqldf("
  SELECT o.service_type,
         COUNT(DISTINCT o.order_id) AS orders,
         COUNT(DISTINCT cp.complaint_id) AS complaints,
         ROUND(COUNT(DISTINCT cp.complaint_id) * 100.0 / COUNT(DISTINCT o.order_id), 1) AS complaint_rate_pct
  FROM orders o
  LEFT JOIN complaints cp ON o.order_id = cp.order_id
  GROUP BY o.service_type
  ORDER BY complaint_rate_pct DESC
")

## 2.7. Drive Training vs. Route Overrides

Determine whether it is undertrained drivers that override routes more often

In [ ]:
sqldf("
  SELECT CASE
           WHEN dr.training_score < 60  THEN 'Low (<60)'
           WHEN dr.training_score <= 75 THEN 'Mid (60-75)'
           ELSE 'High (>75)'
         END AS training_band,
         COUNT(DISTINCT dr.driver_id) AS drivers,
         ROUND(AVG(d.manual_route_override_count), 2) AS avg_overrides
  FROM deliveries d
  JOIN drivers dr ON d.driver_id = dr.driver_id
  GROUP BY training_band
  ORDER BY avg_overrides DESC
")

## 2.8. Incidents by Vehicle Type

Determine incidents per 100 deliveries in each vehicle type

In [ ]:
sqldf("
  SELECT v.vehicle_type,
         COUNT(DISTINCT i.incident_id) AS incidents,
         COUNT(DISTINCT d.delivery_id) AS deliveries,
         ROUND(COUNT(DISTINCT i.incident_id) * 100.0 / COUNT(DISTINCT d.delivery_id), 1) AS incidents_per_100
  FROM deliveries d
  JOIN vehicles v  ON d.vehicle_id = v.vehicle_id
  LEFT JOIN incidents i ON d.delivery_id = i.delivery_id
  GROUP BY v.vehicle_type
  ORDER BY incidents_per_100 DESC
")